# Adaptive Variable-Resolution 2.5D LiDAR Mapping
## Stage 1: Synthetic Prototype (NO real dataset)

> **SCIENTIFIC HONESTY NOTICE — READ FIRST**
> - All point clouds in this notebook are **synthetic, controlled LiDAR-like data** (geometric primitives + noise). They are **NOT real LiDAR**.
> - All perception features are **simulated perception input** (hand-defined), **NOT trained-AI output**.
> - All timings / cell counts are **synthetic benchmark results** measured on this synthetic scene only. **NOT real-world performance.** No FPS/latency/accuracy claims about the real world are made.
> - No SemanticKITTI / nuScenes / external dataset is downloaded or used in this stage.
> - Code is structured so `SyntheticSceneGenerator` can later be replaced by a `NuScenesLiDARLoader` without rewriting the Importance / Resolution / Mapper cores.


## 1. Environment Setup
Matplotlib is the guaranteed visualization fallback. Open3D is optional and the notebook works without it.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from dataclasses import dataclass, field, asdict
from pathlib import Path
import json
import time
import math
import textwrap
from typing import Dict, List, Tuple, Any, Optional

print(f"numpy {np.__version__} | pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
try:
    import open3d as o3d
    print(f"open3d {o3d.__version__} available (optional only)")
    HAS_OPEN3D = True
except Exception as e:
    print(f"open3d unavailable ({e}) -> using Matplotlib fallback (guaranteed)")
    HAS_OPEN3D = False

SEED = 42
rng_global = np.random.default_rng(SEED)
np.random.seed(SEED)
print(f"Deterministic seed set: SEED={SEED}")
print("[CHECK] Environment setup complete.")


## 2. Configuration
All values below are **initial prototype parameters** (tunable, not measured real-world optima).


In [ ]:
# ---- Initial prototype parameters (ALL configurable, ALL synthetic-stage) ----
MAX_RANGE_M: float = 100.0
SEED = 42

# Resolution levels (m): 5 cm / 10 cm / 20 cm / 50 cm
RES_FINE = 0.05
RES_MED_FINE = 0.10
RES_MED_COARSE = 0.20
RES_COARSE = 0.50
RESOLUTION_LEVELS: Dict[str, float] = {
    "fine": RES_FINE,            # 5 cm
    "medium_fine": RES_MED_FINE, # 10 cm
    "medium_coarse": RES_MED_COARSE,  # 20 cm
    "coarse": RES_COARSE,        # 50 cm
}

# Importance -> resolution policy (initial prototype thresholds)
THRESH_FINE = 0.75
THRESH_MED_FINE = 0.50
THRESH_MED_COARSE = 0.25

# Importance weights (initial prototype parameters; must sum to 1)
W: Dict[str, float] = {
    "distance": 0.30,
    "semantic": 0.30,
    "terrain": 0.15,
    "dynamic": 0.15,
    "uncertainty": 0.10,
}
LAMBDA_UNCERTAINTY: float = 0.5  # safety bonus weight for uncertainty

# Class -> semantic importance (simulated perception prior, configurable)
SEMANTIC_IMPORTANCE: Dict[str, float] = {
    "pedestrian": 0.95,
    "vehicle": 0.90,
    "unknown_obstacle": 0.80,
    "building": 0.50,
    "rough_terrain": 0.45,
    "vegetation": 0.35,
    "road": 0.10,
}

# Distance-based baseline bins (synthetic comparison only)
DIST_BASELINE_BINS = [(10.0, 0.05), (30.0, 0.10), (60.0, 0.20), (float("inf"), 0.50)]

def validate_config() -> None:
    assert MAX_RANGE_M > 0, "MAX_RANGE_M must be positive"
    assert all(v > 0 for v in RESOLUTION_LEVELS.values()), "resolutions must be positive"
    assert all(v >= 0 for v in W.values()), "weights must be non-negative"
    s = sum(W.values())
    assert abs(s - 1.0) < 1e-9, f"weights must sum to 1, got {s}"
    assert THRESH_FINE > THRESH_MED_FINE > THRESH_MED_COARSE > 0, "thresholds must be ordered"
    assert 0.0 <= LAMBDA_UNCERTAINTY <= 2.0, "lambda out of sane range"
    for k, v in SEMANTIC_IMPORTANCE.items():
        assert 0.0 <= v <= 1.0, f"semantic importance {k}={v} not in [0,1]"

validate_config()
print("MAX_RANGE_M =", MAX_RANGE_M)
print("RESOLUTION_LEVELS =", RESOLUTION_LEVELS)
print("Policy: I>=0.75 -> 0.05m | I>=0.50 -> 0.10m | I>=0.25 -> 0.20m | else -> 0.50m")
print("W =", W, "| LAMBDA =", LAMBDA_UNCERTAINTY)
print("SEMANTIC_IMPORTANCE =", SEMANTIC_IMPORTANCE)
print("[CHECK] Configuration valid (initial prototype parameters).")


## 3. Synthetic Scene Definition
Structured synthetic environment built from geometric primitives — **NOT real LiDAR**. Every region gets a unique `region_id`.


In [ ]:
# Region specs for the default synthetic scene (positions in meters, synthetic only).
# Format: region_id, semantic_class, kind, center(x,y,z), size, extras
DEFAULT_REGION_SPECS: List[Dict[str, Any]] = [
    {"region_id": 0, "semantic_class": "road", "kind": "plane",
     "center": (0.0, 0.0, 0.0), "size": (60.0, 20.0), "n_points": 4000, "noise": 0.02},
    {"region_id": 1, "semantic_class": "vehicle", "kind": "box",
     "center": (20.0, 3.0, 0.9), "size": (4.4, 1.8, 1.6), "n_points": 2000, "noise": 0.03},
    {"region_id": 2, "semantic_class": "pedestrian", "kind": "pedestrian",
     "center": (30.0, -2.0, 0.9), "size": (0.5, 0.5, 1.8), "n_points": 800, "noise": 0.02},
    {"region_id": 3, "semantic_class": "building", "kind": "wall",
     "center": (15.0, -12.0, 3.0), "size": (40.0, 0.5, 6.0), "n_points": 2500, "noise": 0.03},
    {"region_id": 4, "semantic_class": "vegetation", "kind": "scatter",
     "center": (-10.0, 8.0, 1.0), "size": (10.0, 6.0, 2.5), "n_points": 1200, "noise": 0.15},
    {"region_id": 5, "semantic_class": "rough_terrain", "kind": "rough",
     "center": (-20.0, -5.0, 0.0), "size": (16.0, 10.0), "n_points": 2000, "noise": 0.05, "rough_amp": 0.35},
    {"region_id": 6, "semantic_class": "unknown_obstacle", "kind": "box",
     "center": (45.0, 1.0, 0.6), "size": (1.5, 1.5, 1.2), "n_points": 900, "noise": 0.06},
    # USP pair: pedestrian at ~70 m and empty road patch at ~70 m (same distance, different semantics)
    {"region_id": 7, "semantic_class": "pedestrian", "kind": "pedestrian",
     "center": (70.0, 2.0, 0.9), "size": (0.5, 0.5, 1.8), "n_points": 800, "noise": 0.03},
    {"region_id": 8, "semantic_class": "road", "kind": "plane",
     "center": (70.0, -6.0, 0.0), "size": (8.0, 4.0), "n_points": 1200, "noise": 0.02},
]
print(f"Defined {len(DEFAULT_REGION_SPECS)} synthetic regions (controlled primitives, NOT real data).")
for s in DEFAULT_REGION_SPECS:
    print(f"  id={s['region_id']} class={s['semantic_class']:16s} kind={s['kind']:10s} center={s['center']}")


## 4. Synthetic LiDAR Point Generation
Generates an N x 4 `(x, y, z, intensity)` array per region. Controlled synthetic input only — not fake results.


In [ ]:
class SyntheticSceneGenerator:
    """Generate controlled synthetic LiDAR-like points from geometric primitives.

    Replaceable later by NuScenesLiDARLoader returning the same region dict format.
    No real dataset is used here.
    """
    def __init__(self, seed: int = SEED, dropout: float = 0.02):
        assert 0.0 <= dropout < 1.0, "dropout must be in [0,1)"
        self.seed = seed
        self.dropout = dropout
        self.rng = np.random.default_rng(seed)

    def _dropout(self, pts: np.ndarray) -> np.ndarray:
        if self.dropout <= 0 or len(pts) == 0:
            return pts
        keep = self.rng.random(len(pts)) >= self.dropout
        return pts[keep]

    def _plane(self, cx, cy, cz, sx, sy, n, noise) -> np.ndarray:
        x = self.rng.uniform(cx - sx/2, cx + sx/2, n)
        y = self.rng.uniform(cy - sy/2, cy + sy/2, n)
        z = cz + self.rng.normal(0, noise, n)
        inten = np.clip(self.rng.normal(0.35, 0.08, n), 0, 1)
        return np.column_stack([x, y, z, inten])

    def _box(self, cx, cy, cz, sx, sy, sz, n, noise) -> np.ndarray:
        # sample on box surfaces (top + 4 sides) to look like a LiDAR return
        faces = ["top"]*max(1, n//3) + ["side"]*(n - max(1, n//3))
        xs, ys, zs = [], [], []
        for f in faces:
            if f == "top":
                xs.append(self.rng.uniform(cx-sx/2, cx+sx/2)); ys.append(self.rng.uniform(cy-sy/2, cy+sy/2)); zs.append(cz+sz/2)
            else:
                side = self.rng.integers(0, 4)
                if side == 0: xs.append(cx+sx/2); ys.append(self.rng.uniform(cy-sy/2, cy+sy/2)); zs.append(self.rng.uniform(cz-sz/2, cz+sz/2))
                elif side == 1: xs.append(cx-sx/2); ys.append(self.rng.uniform(cy-sy/2, cy+sy/2)); zs.append(self.rng.uniform(cz-sz/2, cz+sz/2))
                elif side == 2: xs.append(self.rng.uniform(cx-sx/2, cx+sx/2)); ys.append(cy+sy/2); zs.append(self.rng.uniform(cz-sz/2, cz+sz/2))
                else: xs.append(self.rng.uniform(cx-sx/2, cx+sx/2)); ys.append(cy-sy/2); zs.append(self.rng.uniform(cz-sz/2, cz+sz/2))
        pts = np.column_stack([np.array(xs), np.array(ys), np.array(zs)]) + self.rng.normal(0, noise, (len(xs), 3))
        inten = np.clip(self.rng.normal(0.6, 0.12, len(pts)), 0, 1)
        return np.column_stack([pts, inten])

    def _pedestrian(self, cx, cy, cz, sx, sy, sz, n, noise) -> np.ndarray:
        # compact vertical cluster: torso + head blob
        z_body = self.rng.normal(cz, sz*0.28, n)
        x = cx + self.rng.normal(0, sx*0.35, n)
        y = cy + self.rng.normal(0, sy*0.35, n)
        x += self.rng.normal(0, noise, n); y += self.rng.normal(0, noise, n); z_body += self.rng.normal(0, noise, n)
        inten = np.clip(self.rng.normal(0.55, 0.1, n), 0, 1)
        return np.column_stack([x, y, z_body, inten])

    def _wall(self, cx, cy, cz, sx, sy, sz, n, noise) -> np.ndarray:
        x = self.rng.uniform(cx - sx/2, cx + sx/2, n)
        y = cy + self.rng.normal(0, max(noise, sy/2*0.2), n)
        z = self.rng.uniform(cz - sz/2, cz + sz/2, n)
        x += self.rng.normal(0, noise, n); z += self.rng.normal(0, noise, n)
        inten = np.clip(self.rng.normal(0.5, 0.1, n), 0, 1)
        return np.column_stack([x, y, z, inten])

    def _scatter(self, cx, cy, cz, sx, sy, sz, n, noise) -> np.ndarray:
        x = self.rng.normal(cx, sx*0.25, n); y = self.rng.normal(cy, sy*0.25, n); z = np.abs(self.rng.normal(cz*0.5, sz*0.3, n))
        x += self.rng.normal(0, noise, n); y += self.rng.normal(0, noise, n); z += self.rng.normal(0, noise, n)
        inten = np.clip(self.rng.normal(0.3, 0.12, n), 0, 1)
        return np.column_stack([x, y, z, inten])

    def _rough(self, cx, cy, cz, sx, sy, n, noise, rough_amp=0.35) -> np.ndarray:
        x = self.rng.uniform(cx - sx/2, cx + sx/2, n)
        y = self.rng.uniform(cy - sy/2, cy + sy/2, n)
        z = cz + rough_amp*np.sin(0.8*x)*np.cos(0.9*y) + self.rng.normal(0, noise + 0.08, n)
        inten = np.clip(self.rng.normal(0.32, 0.08, n), 0, 1)
        return np.column_stack([x, y, z, inten])

    def generate_region_points(self, spec: Dict[str, Any]) -> np.ndarray:
        kind = spec["kind"]; cx, cy, cz = spec["center"]; n = int(spec.get("n_points", 1000)); noise = float(spec.get("noise", 0.02))
        assert n > 0, "n_points must be positive"
        sx, sy = spec["size"][0], spec["size"][1]
        sz = spec["size"][2] if len(spec["size"]) > 2 else 1.0
        if kind == "plane": pts = self._plane(cx, cy, cz, sx, sy, n, noise)
        elif kind == "box": pts = self._box(cx, cy, cz, sx, sy, sz, n, noise)
        elif kind == "pedestrian": pts = self._pedestrian(cx, cy, cz, sx, sy, sz, n, noise)
        elif kind == "wall": pts = self._wall(cx, cy, cz, sx, sy, sz, n, noise)
        elif kind == "scatter": pts = self._scatter(cx, cy, cz, sx, sy, sz, n, noise)
        elif kind == "rough": pts = self._rough(cx, cy, cz, sx, sy, n, noise, spec.get("rough_amp", 0.35))
        else: raise ValueError(f"unknown primitive kind: {kind}")
        pts = self._dropout(pts)
        assert pts.ndim == 2 and pts.shape[1] == 4, "points must be N x 4"
        assert np.all(np.isfinite(pts)), "points must be finite"
        return pts

    def generate_scene(self, specs: List[Dict[str, Any]]) -> Dict[int, np.ndarray]:
        out: Dict[int, np.ndarray] = {}
        for s in specs:
            out[int(s["region_id"])] = self.generate_region_points(s)
        return out

gen = SyntheticSceneGenerator(seed=SEED, dropout=0.02)
clouds = gen.generate_scene(DEFAULT_REGION_SPECS)
total = sum(len(v) for v in clouds.values())
print(f"[SYNTHETIC INPUT] Generated {len(clouds)} regions, {total} points total (controlled synthetic data).")
for rid, pts in clouds.items():
    print(f"  region {rid}: N={len(pts)} xyz_mean={pts[:, :3].mean(0).round(2)}")


## 5. Synthetic Semantic / Region Features
Simulated perception input (hand-defined priors) — **NOT trained-AI output**. Configurable class mapping.


In [ ]:
# Simulated perception priors per region (synthetic-stage stand-ins for a real detector).
# distance_m is computed from region center range; the rest are controlled test inputs.
SIM_REGION_FEATURES: Dict[int, Dict[str, float]] = {
    0: {"confidence": 0.95, "terrain_complexity": 0.05, "dynamic_relevance": 0.00, "uncertainty": 0.05},
    1: {"confidence": 0.92, "terrain_complexity": 0.15, "dynamic_relevance": 0.80, "uncertainty": 0.10},
    2: {"confidence": 0.88, "terrain_complexity": 0.20, "dynamic_relevance": 0.95, "uncertainty": 0.15},
    3: {"confidence": 0.90, "terrain_complexity": 0.10, "dynamic_relevance": 0.05, "uncertainty": 0.10},
    4: {"confidence": 0.70, "terrain_complexity": 0.55, "dynamic_relevance": 0.05, "uncertainty": 0.35},
    5: {"confidence": 0.75, "terrain_complexity": 0.90, "dynamic_relevance": 0.05, "uncertainty": 0.25},
    6: {"confidence": 0.40, "terrain_complexity": 0.50, "dynamic_relevance": 0.40, "uncertainty": 0.90},
    7: {"confidence": 0.85, "terrain_complexity": 0.20, "dynamic_relevance": 0.95, "uncertainty": 0.20},
    8: {"confidence": 0.95, "terrain_complexity": 0.05, "dynamic_relevance": 0.00, "uncertainty": 0.05},
}

def build_region_features(specs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    feats = []
    for s in specs:
        rid = int(s["region_id"])
        cx, cy, _ = s["center"]
        dist = float(np.hypot(cx, cy))
        sem = str(s["semantic_class"])
        assert sem in SEMANTIC_IMPORTANCE, f"unknown semantic class {sem}"
        f = SIM_REGION_FEATURES.get(rid, {"confidence": 0.8, "terrain_complexity": 0.2, "dynamic_relevance": 0.2, "uncertainty": 0.2})
        feats.append({
            "region_id": rid, "semantic_class": sem, "confidence": float(f["confidence"]),
            "distance_m": dist, "terrain_complexity": float(f["terrain_complexity"]),
            "dynamic_relevance": float(f["dynamic_relevance"]), "uncertainty": float(f["uncertainty"]),
            "semantic_importance": float(SEMANTIC_IMPORTANCE[sem]),
        })
    return feats

region_features = build_region_features(DEFAULT_REGION_SPECS)
df_feat = pd.DataFrame(region_features)
print("[SYNTHETIC SIMULATED PERCEPTION] (hand-defined, NOT model predictions)")
print(df_feat.to_string(index=False))


## 6. Data Structures
Strongly typed dataclasses with validation.


In [ ]:
@dataclass
class SyntheticRegion:
    region_id: int
    semantic_class: str
    points: np.ndarray
    distance_m: float
    semantic_importance: float
    terrain_complexity: float
    dynamic_relevance: float
    uncertainty: float
    confidence: float
    def __post_init__(self):
        assert self.points.ndim == 2 and self.points.shape[1] == 4, "points must be N x 4"
        assert self.distance_m >= 0 and np.isfinite(self.distance_m), "invalid distance"
        for name in ["semantic_importance", "terrain_complexity", "dynamic_relevance", "uncertainty", "confidence"]:
            v = getattr(self, name)
            assert isinstance(v, (int, float)) and np.isfinite(v) and 0.0 <= v <= 1.0, f"{name}={v} not in [0,1]"
        assert self.semantic_class in SEMANTIC_IMPORTANCE, f"unknown class {self.semantic_class}"

@dataclass
class ImportanceResult:
    region_id: int
    distance_score: float
    semantic_score: float
    terrain_score: float
    dynamic_score: float
    uncertainty_score: float
    base_importance: float
    safe_importance: float
    selected_resolution_m: float
    resolution_level: str

@dataclass
class MapCell:
    x: float
    y: float
    elevation: float
    elevation_std: float
    occupancy: int
    semantic_class: str
    confidence: float
    importance: float
    resolution: float
    region_id: int
    point_count: int

@dataclass
class MapResult:
    cells: List[MapCell]
    importance: List[ImportanceResult]
    elapsed_s: float
    n_points: int
    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([asdict(c) for c in self.cells])

def assemble_regions(specs, clouds, feats) -> List[SyntheticRegion]:
    feat_by_id = {f["region_id"]: f for f in feats}
    regs = []
    for s in specs:
        rid = int(s["region_id"]); f = feat_by_id[rid]
        regs.append(SyntheticRegion(rid, str(s["semantic_class"]), clouds[rid],
            float(f["distance_m"]), float(f["semantic_importance"]), float(f["terrain_complexity"]),
            float(f["dynamic_relevance"]), float(f["uncertainty"]), float(f["confidence"])))
    return regs

regions = assemble_regions(DEFAULT_REGION_SPECS, clouds, region_features)
print(f"[CHECK] Assembled {len(regions)} validated SyntheticRegion objects.")


## 7. Importance Engine
Formula: `I_base = wd*D + ws*S + wt*T + wm*M + wu*U` with `D = clip(1 - d/Rmax, 0, 1)`, then safety modifier `I_safe = min(1, I_base + lambda*U)`.

> Uncertainty is treated as a reason to preserve more information rather than aggressively reduce resolution.


In [ ]:
class ImportanceEngine:
    """Weighted importance fusion + uncertainty safety bonus (synthetic prototype)."""
    def __init__(self, weights: Dict[str, float] = W, max_range_m: float = MAX_RANGE_M,
                 lambda_uncertainty: float = LAMBDA_UNCERTAINTY):
        assert max_range_m > 0, "max_range_m must be positive"
        assert all(v >= 0 for v in weights.values()), "weights must be non-negative"
        assert abs(sum(weights.values()) - 1.0) < 1e-9, "weights must sum to 1"
        self.w = dict(weights); self.max_range_m = max_range_m; self.lam = lambda_uncertainty

    @staticmethod
    def _clip01(v: float) -> float:
        return float(min(1.0, max(0.0, v)))

    def distance_score(self, distance_m: float) -> float:
        assert np.isfinite(distance_m) and distance_m >= 0, "invalid distance"
        return self._clip01(1.0 - distance_m / self.max_range_m)

    def score_region(self, r: SyntheticRegion) -> ImportanceResult:
        D = self.distance_score(r.distance_m)
        S = self._clip01(r.semantic_importance); T = self._clip01(r.terrain_complexity)
        M = self._clip01(r.dynamic_relevance); U = self._clip01(r.uncertainty)
        base = self.w["distance"]*D + self.w["semantic"]*S + self.w["terrain"]*T + self.w["dynamic"]*M + self.w["uncertainty"]*U
        safe = min(1.0, base + self.lam * U)
        return ImportanceResult(r.region_id, D, S, T, M, U, self._clip01(base), self._clip01(safe), -1.0, "")

imp_engine = ImportanceEngine()
for r in regions:
    s = imp_engine.score_region(r)
    print(f"region {r.region_id} ({r.semantic_class:16s} d={r.distance_m:5.1f}m): D={s.distance_score:.2f} S={s.semantic_score:.2f} "
          f"T={s.terrain_score:.2f} M={s.dynamic_score:.2f} U={s.uncertainty_score:.2f} -> I_base={s.base_importance:.3f} I_safe={s.safe_importance:.3f}")
print("[CHECK] ImportanceEngine working.")


## 8. Adaptive Resolution Engine
`0.75-1.00 -> 5cm | 0.50-0.75 -> 10cm | 0.25-0.50 -> 20cm | 0.00-0.25 -> 50cm`. Boundaries inclusive of lower bound.


In [ ]:
class ResolutionEngine:
    """Map importance in [0,1] to a concrete cell size. Boundary-explicit."""
    def __init__(self, t_fine=THRESH_FINE, t_med_fine=THRESH_MED_FINE, t_med_coarse=THRESH_MED_COARSE,
                 levels: Dict[str, float] = RESOLUTION_LEVELS):
        assert t_fine > t_med_fine > t_med_coarse > 0
        assert all(v > 0 for v in levels.values())
        self.t_fine, self.t_med_fine, self.t_med_coarse = t_fine, t_med_fine, t_med_coarse
        self.levels = dict(levels)
    def select(self, importance: float) -> Tuple[float, str]:
        assert np.isfinite(importance) and 0.0 <= importance <= 1.0, "importance must be in [0,1]"
        if importance >= self.t_fine: return self.levels["fine"], "fine (5 cm)"
        if importance >= self.t_med_fine: return self.levels["medium_fine"], "medium_fine (10 cm)"
        if importance >= self.t_med_coarse: return self.levels["medium_coarse"], "medium_coarse (20 cm)"
        return self.levels["coarse"], "coarse (50 cm)"

res_engine = ResolutionEngine()
# attach resolutions to importance results for the default scene
imp_results: List[ImportanceResult] = []
for r in regions:
    s = imp_engine.score_region(r)
    res_m, lvl = res_engine.select(s.safe_importance)
    s.selected_resolution_m, s.resolution_level = res_m, lvl
    imp_results.append(s)
print(pd.DataFrame([asdict(s) for s in imp_results])[["region_id","base_importance","safe_importance","selected_resolution_m","resolution_level"]].to_string(index=False))
print("[CHECK] ResolutionEngine working.")


## 9. Variable-Resolution 2.5D Mapper
**True variable resolution**: each region is binned with its OWN cell size (separate region-specific grids), then merged. NOT a colored uniform grid.


In [ ]:
class VariableResolutionMapper2_5D:
    """Region-specific XY binning at each region's selected resolution + height statistics."""
    def __init__(self, imp_engine: ImportanceEngine, res_engine: ResolutionEngine):
        self.imp = imp_engine; self.res = res_engine
    def map_region(self, region: SyntheticRegion, importance: float) -> List[MapCell]:
        res_m, _ = self.res.select(importance)
        pts = region.points
        assert len(pts) > 0, f"region {region.region_id} has no points"
        ix = np.floor(pts[:, 0] / res_m).astype(np.int64)
        iy = np.floor(pts[:, 1] / res_m).astype(np.int64)
        keys = np.column_stack([ix, iy])
        ukeys, inverse = np.unique(keys, axis=0, return_inverse=True)
        cells: List[MapCell] = []
        for k, (cx, cy) in enumerate(ukeys):
            sel = inverse == k
            z = pts[sel, 2]
            cells.append(MapCell(
                x=float((cx + 0.5) * res_m), y=float((cy + 0.5) * res_m),
                elevation=float(z.mean()), elevation_std=float(z.std() if len(z) > 1 else 0.0),
                occupancy=1, semantic_class=region.semantic_class, confidence=region.confidence,
                importance=float(importance), resolution=float(res_m),
                region_id=region.region_id, point_count=int(sel.sum())))
        return cells
    def map_scene(self, regs: List[SyntheticRegion]) -> MapResult:
        t0 = time.perf_counter()
        all_cells: List[MapCell] = []; imp_list: List[ImportanceResult] = []; n_pts = 0
        for r in regs:
            s = self.imp.score_region(r)
            res_m, lvl = self.res.select(s.safe_importance)
            s.selected_resolution_m, s.resolution_level = res_m, lvl
            all_cells.extend(self.map_region(r, s.safe_importance))
            imp_list.append(s); n_pts += len(r.points)
        return MapResult(all_cells, imp_list, time.perf_counter() - t0, n_pts)

mapper = VariableResolutionMapper2_5D(imp_engine, res_engine)
map_result = mapper.map_scene(regions)
print(f"[SYNTHETIC MAP] {len(map_result.cells)} cells from {map_result.n_points} points in {map_result.elapsed_s*1000:.1f} ms")
print(f"Resolutions present: {sorted(set(c.resolution for c in map_result.cells))}")
assert len(set(c.resolution for c in map_result.cells)) > 1, "expected multiple resolutions"
print("[CHECK] Adaptive 2.5D Mapper working — different cell sizes actually created.")


## 10. Complete Pipeline
`run_pipeline(scene)` returns structured results.


In [ ]:
def run_pipeline(specs: List[Dict[str, Any]], seed: int = SEED, dropout: float = 0.02) -> Dict[str, Any]:
    """Synthetic scene -> features -> importance -> resolution -> adaptive 2.5D map."""
    t0 = time.perf_counter()
    g = SyntheticSceneGenerator(seed=seed, dropout=dropout)
    cl = g.generate_scene(specs)
    feats = build_region_features(specs)
    regs = assemble_regions(specs, cl, feats)
    ie = ImportanceEngine(); re = ResolutionEngine()
    mp = VariableResolutionMapper2_5D(ie, re)
    mr = mp.map_scene(regs)
    return {"regions": regs, "map": mr, "clouds": cl, "features": feats,
            "elapsed_s": time.perf_counter() - t0, "synthetic": True}

result = run_pipeline(DEFAULT_REGION_SPECS)
print(f"Pipeline done: {len(result['map'].cells)} cells, {result['map'].n_points} pts, {result['elapsed_s']*1000:.1f} ms (synthetic).")
print("[CHECK] Complete pipeline working.")


## 11. Core USP Demonstration
**Resolution is driven by importance, not distance alone.** Same distance (~70 m) must yield different resolutions.


In [ ]:
# USP pair: region 7 (pedestrian @ ~70m) vs region 8 (empty road @ ~70m) — computed, NOT hard-coded.
usp = run_pipeline([s for s in DEFAULT_REGION_SPECS if s["region_id"] in (7, 8)])
rows = []
for r, s in zip(usp["regions"], usp["map"].importance):
    rows.append({"Region": f"{r.region_id}:{r.semantic_class}", "Distance_m": round(r.distance_m, 2),
                 "Semantic": round(r.semantic_importance, 2), "Dynamic": round(r.dynamic_relevance, 2),
                 "Importance": round(s.safe_importance, 3), "Resolution_m": s.selected_resolution_m,
                 "Level": s.resolution_level,
                 "Cells": sum(1 for c in usp["map"].cells if c.region_id == r.region_id)})
df_usp = pd.DataFrame(rows)
print("SYNTHETIC USP EXPERIMENT (computed by the engines):")
print(df_usp.to_string(index=False))
d = abs(usp["regions"][0].distance_m - usp["regions"][1].distance_m)
print(f"|d_ped - d_road| = {d:.2f} m (same/nearly identical distance)")
ped_res = df_usp.loc[df_usp["Region"].str.contains("pedestrian"), "Resolution_m"].iloc[0]
road_res = df_usp.loc[df_usp["Region"].str.contains("road"), "Resolution_m"].iloc[0]
assert d < 12.0, "USP pair must be at nearly identical distance"
assert ped_res < road_res, "USP FAILED: pedestrian must get finer resolution than empty road at same distance"
print(f"\nUSP VERIFIED: pedestrian {ped_res} m < empty-road {road_res} m at SAME distance -> importance, not distance, drove resolution.")


## 12. Scenario Tests
8 synthetic scenarios (controlled inputs, synthetic outputs).


In [ ]:
def _spec(rid: int, **overrides) -> Dict[str, Any]:
    """Copy a default region spec by id, so region_id -> simulated-feature mapping stays consistent."""
    s = dict(next(s for s in DEFAULT_REGION_SPECS if s["region_id"] == rid))
    s.update(overrides)
    return s

def scenario_specs(kind: str) -> List[Dict[str, Any]]:
    if kind == "near_ped": return [_spec(0), _spec(2, center=(8.0, 1.0, 0.9))]
    if kind == "far_ped": return [_spec(0), _spec(7)]
    if kind == "near_vehicle": return [_spec(0), _spec(1, center=(12.0, 2.0, 0.9))]
    if kind == "far_road": return [_spec(8)]
    if kind == "rough": return [_spec(0), _spec(5)]
    if kind == "unknown": return [_spec(0), _spec(6)]
    if kind == "dense": return DEFAULT_REGION_SPECS
    if kind == "empty": return [_spec(0)]
    raise ValueError(kind)

SCENARIOS = ["near_ped", "far_ped", "near_vehicle", "far_road", "rough", "unknown", "dense", "empty"]
scenario_rows = []
for sc in SCENARIOS:
    specs = scenario_specs(sc)
    # patch SIM_REGION_FEATURES lookup by remapping scenario-local ids onto sensible priors
    out = run_pipeline(specs)
    for r, s in zip(out["regions"], out["map"].importance):
        scenario_rows.append({"scenario": sc, "region": f"{r.region_id}:{r.semantic_class}",
            "importance": round(s.safe_importance, 3), "resolution_m": s.selected_resolution_m,
            "cells": sum(1 for c in out["map"].cells if c.region_id == r.region_id)})
df_scen = pd.DataFrame(scenario_rows)
print("[SYNTHETIC SCENARIO TESTS]")
print(df_scen.to_string(index=False))
print("[CHECK] Scenario tests complete.")


## 13. Unit Tests
Executable tests; all must pass.


In [ ]:
passed, failed = 0, 0
def check(name, cond):
    global passed, failed
    if cond: passed += 1; print(f"[PASS] {name}")
    else: failed += 1; print(f"[FAIL] {name}")

ie_t = ImportanceEngine(); re_t = ResolutionEngine()
# normalization
check("D(0)==1", ie_t.distance_score(0) == 1.0)
check("D(50)==0.5", abs(ie_t.distance_score(50) - 0.5) < 1e-9)
check("D(100)==0", ie_t.distance_score(100) == 0.0)
check("D(150) clipped to 0", ie_t.distance_score(150) == 0.0)
# importance bounds
rr = SyntheticRegion(99, "road", np.zeros((5, 4)), 10.0, 0.1, 0.1, 0.0, 0.05, 0.9)
s = ie_t.score_region(rr)
check("0<=I<=1", 0.0 <= s.safe_importance <= 1.0 and 0.0 <= s.base_importance <= 1.0)
# resolution mapping
check("0.90->5cm", re_t.select(0.90)[0] == 0.05)
check("0.60->10cm", re_t.select(0.60)[0] == 0.10)
check("0.30->20cm", re_t.select(0.30)[0] == 0.20)
check("0.10->50cm", re_t.select(0.10)[0] == 0.50)
check("boundary 0.75->5cm", re_t.select(0.75)[0] == 0.05)
check("boundary 0.50->10cm", re_t.select(0.50)[0] == 0.10)
check("boundary 0.25->20cm", re_t.select(0.25)[0] == 0.20)
# invalid inputs
for name, fn in [
    ("negative distance raises", lambda: ie_t.distance_score(-1)),
    ("NaN distance raises", lambda: ie_t.distance_score(float("nan"))),
    ("inf distance raises", lambda: ie_t.distance_score(float("inf"))),
    ("bad weights raise", lambda: ImportanceEngine(weights={"distance": 0.5, "semantic": 0.5, "terrain": 0.0, "dynamic": 0.5, "uncertainty": 0.5})),
    ("negative resolution raises", lambda: ResolutionEngine(levels={"fine": -0.05, "medium_fine": 0.1, "medium_coarse": 0.2, "coarse": 0.5})),
]:
    try:
        fn(); check(name, False)
    except (AssertionError, ValueError):
        check(name, True)
# mapper
mp_t = VariableResolutionMapper2_5D(ie_t, re_t)
cells_hi = mp_t.map_region(SyntheticRegion(1, "pedestrian", clouds[7], 70.0, 0.95, 0.2, 0.95, 0.2, 0.85), 0.9)
cells_lo = mp_t.map_region(SyntheticRegion(8, "road", clouds[8], 70.0, 0.1, 0.05, 0.0, 0.05, 0.95), 0.1)
req = {"x","y","elevation","occupancy","semantic_class","confidence","importance","resolution","region_id","point_count"}
check("cells have required fields", all(req <= set(asdict(c).keys()) for c in cells_hi + cells_lo))
check("cells have valid resolution", all(c.resolution in (0.05, 0.10, 0.20, 0.50) for c in cells_hi + cells_lo))
check("different importance -> different resolution", cells_hi[0].resolution != cells_lo[0].resolution)
print(f"\nUnit tests: {passed} passed, {failed} failed")
assert failed == 0, "unit tests failed"
print("[CHECK] All unit tests passed.")


## 14. Visualization
All figures use synthetic data only. Fig.3 draws ACTUAL variable-size cells.


In [ ]:
RESULTS_DIR = Path("results"); VIZ_DIR = RESULTS_DIR / "visualizations"
RESULTS_DIR.mkdir(parents=True, exist_ok=True); VIZ_DIR.mkdir(parents=True, exist_ok=True)

CLASS_COLORS = {"road": "#9e9e9e", "vehicle": "#1f77b4", "pedestrian": "#d62728", "building": "#ff7f0e",
                "vegetation": "#2ca02c", "rough_terrain": "#8c564b", "unknown_obstacle": "#9467bd"}

# V1: synthetic 3D scene (actual generated points, subsampled for speed)
fig = plt.figure(figsize=(11, 4.5))
ax = fig.add_subplot(121, projection="3d")
for r in regions:
    p = r.points; idx = np.random.default_rng(SEED).choice(len(p), min(800, len(p)), replace=False)
    ax.scatter(p[idx,0], p[idx,1], p[idx,2], s=2, color=CLASS_COLORS.get(r.semantic_class, "k"), label=f"{r.region_id}:{r.semantic_class}")
ax.set_title("V1: Synthetic 3D scene (synthetic primitives, NOT real LiDAR)"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.legend(fontsize=7, loc="upper left")
ax2 = fig.add_subplot(122)
for r in regions:
    p = r.points; idx = np.random.default_rng(SEED+1).choice(len(p), min(1500, len(p)), replace=False)
    ax2.scatter(p[idx,0], p[idx,1], s=2, color=CLASS_COLORS.get(r.semantic_class, "k"), label=f"{r.region_id}:{r.semantic_class}")
ax2.set_title("BEV of synthetic scene"); ax2.set_aspect("equal"); ax2.set_xlabel("x [m]"); ax2.set_ylabel("y [m]")
plt.tight_layout(); plt.savefig(VIZ_DIR/"v1_synthetic_3d_scene.png", dpi=150); plt.show()

# V2: importance heatmap
df_imp = pd.DataFrame([{"region": f"{r.region_id}:{r.semantic_class}", "I": s.safe_importance} for r, s in zip(regions, imp_results)]).sort_values("I")
plt.figure(figsize=(8, 3.5)); plt.barh(df_imp["region"], df_imp["I"], color=plt.cm.inferno(df_imp["I"].values))
plt.xlabel("safe importance"); plt.title("V2: Region importance (synthetic experiment)"); plt.tight_layout()
plt.savefig(VIZ_DIR/"v2_importance_heatmap.png", dpi=150); plt.show()

# V3: adaptive resolution map — TRUE variable cell sizes
fig, ax = plt.subplots(figsize=(11, 6))
drawn = 0
for c in map_result.cells:
    if drawn > 6000: break
    ax.add_patch(patches.Rectangle((c.x - c.resolution/2, c.y - c.resolution/2), c.resolution, c.resolution,
        facecolor=CLASS_COLORS.get(c.semantic_class, "gray"), edgecolor="k", lw=0.15, alpha=0.8))
    drawn += 1
ax.set_title(f"V3: Adaptive resolution map — ACTUAL cell sizes (5/10/20/50 cm, {len(map_result.cells)} cells, synthetic)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_aspect("equal"); ax.autoscale_view(); plt.tight_layout()
plt.savefig(VIZ_DIR/"v3_adaptive_resolution_map.png", dpi=150); plt.show()

# V4: 2.5D elevation map (X, Y, Elevation)
df_cells = map_result.to_dataframe()
plt.figure(figsize=(9, 5))
sc = plt.scatter(df_cells["x"], df_cells["y"], c=df_cells["elevation"], cmap="terrain", s=4)
plt.colorbar(sc, label="elevation [m]"); plt.gca().set_aspect("equal")
plt.title("V4: 2.5D elevation map (cell mean height, synthetic)"); plt.xlabel("x [m]"); plt.ylabel("y [m]")
plt.tight_layout(); plt.savefig(VIZ_DIR/"v4_elevation_map.png", dpi=150); plt.show()

# V5: side-by-side Uniform 5cm vs Distance-based vs Proposed (cell-count proxy on same scene)
def cells_for_policy(policy):
    n = 0; resolutions = []
    for r in regions:
        res = policy(r)
        ix = np.floor(r.points[:,0]/res).astype(np.int64); iy = np.floor(r.points[:,1]/res).astype(np.int64)
        n += len(np.unique(np.column_stack([ix, iy]), axis=0)); resolutions.append(res)
    return n, float(np.mean(resolutions))
def dist_policy(r):
    d = r.distance_m
    for thr, res in DIST_BASELINE_BINS:
        if d <= thr: return res
    return 0.50
n_uni, m_uni = cells_for_policy(lambda r: 0.05)
n_dst, m_dst = cells_for_policy(dist_policy)
n_pro, m_pro = cells_for_policy(lambda r: next(s.selected_resolution_m for s in imp_results if s.region_id == r.region_id))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(["Uniform 5cm", "Distance-based", "Proposed"], [n_uni, n_dst, n_pro], color=["#7f7f7f", "#1f77b4", "#d62728"])
ax.set_ylabel("# cells"); ax.set_title("V5: Cell count by method (SAME synthetic scene — lower = coarser, not 'better')")
for i, v in enumerate([n_uni, n_dst, n_pro]): ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout(); plt.savefig(VIZ_DIR/"v5_method_comparison.png", dpi=150); plt.show()
print(f"Uniform: {n_uni} cells | Distance: {n_dst} cells | Proposed: {n_pro} cells (synthetic counts)")
print("[CHECK] Visualizations saved to results/visualizations/.")


## 15. Baseline Comparison
Three methods on the SAME synthetic scene. No 'better' claim unless the numbers show it.


In [ ]:
def distance_resolution(d: float) -> float:
    for thr, res in DIST_BASELINE_BINS:
        if d <= thr: return res
    return 0.50

def evaluate_policy(name: str, policy) -> Dict[str, Any]:
    t0 = time.perf_counter(); total_cells = 0; res_list = []; crit = {}
    for r in regions:
        res = policy(r)
        ix = np.floor(r.points[:,0]/res).astype(np.int64); iy = np.floor(r.points[:,1]/res).astype(np.int64)
        nc = len(np.unique(np.column_stack([ix, iy]), axis=0))
        total_cells += nc; res_list.append(res)
        if r.semantic_class in ("pedestrian", "vehicle", "unknown_obstacle"):
            crit[r.region_id] = res
    return {"method": name, "cells": total_cells, "mean_res": float(np.mean(res_list)),
            "critical": crit, "time_ms": (time.perf_counter()-t0)*1000, "points": sum(len(r.points) for r in regions)}

b_uni = evaluate_policy("uniform_5cm", lambda r: 0.05)
b_dst = evaluate_policy("distance_based", lambda r: distance_resolution(r.distance_m))
b_pro = evaluate_policy("proposed", lambda r: next(s.selected_resolution_m for s in imp_results if s.region_id == r.region_id))
df_bench = pd.DataFrame([{**b, "critical": json.dumps(b["critical"])} for b in [b_uni, b_dst, b_pro]])
print("[SYNTHETIC BENCHMARK — same scene, controlled inputs]")
print(df_bench.to_string(index=False))
reduction = 100*(b_uni["cells"] - b_pro["cells"])/b_uni["cells"]
print(f"\nCell reduction (proposed vs uniform, synthetic): {reduction:.1f}% — a synthetic cell-count fact, NOT a real-world speedup claim.")
print("[CHECK] Baselines implemented and compared.")


## 16. Synthetic Performance Measurements
Measured only (timing + counts on synthetic data). Labeled synthetic — no real-time claim.


In [ ]:
rows = []
for b in [b_uni, b_dst, b_pro]:
    rows.append({"method": b["method"], "processing_ms": round(b["time_ms"], 2), "n_cells": b["cells"],
        "n_points": b["points"], "mean_res_m": round(b["mean_res"], 3),
        "critical_res": json.dumps(b["critical"])})
df_perf = pd.DataFrame(rows)
print("[SYNTHETIC BENCHMARK RESULTS — NOT real-world performance, no real-time claim]")
print(df_perf.to_string(index=False))
print("[CHECK] Synthetic performance measured (timing + counts only).")


## 17. Sensitivity Analysis (prototype sensitivity experiments)


In [ ]:
# Vary weights / lambda / thresholds, observe importance -> resolution -> cell count (synthetic).
base_regs = regions
tests = []
# (a) weight variants
for name, wv in {
    "default": W,
    "semantic_heavy": {"distance": 0.15, "semantic": 0.55, "terrain": 0.10, "dynamic": 0.10, "uncertainty": 0.10},
    "distance_heavy": {"distance": 0.60, "semantic": 0.10, "terrain": 0.10, "dynamic": 0.10, "uncertainty": 0.10},
}.items():
    ie = ImportanceEngine(weights=wv); re = ResolutionEngine(); mp = VariableResolutionMapper2_5D(ie, re)
    mr = mp.map_scene(base_regs)
    tests.append({"variant": f"weights:{name}", "mean_I": round(float(np.mean([s.safe_importance for s in mr.importance])), 3),
                  "cells": len(mr.cells), "mean_res": round(float(np.mean([c.resolution for c in mr.cells])), 3)})
# (b) lambda variants
for lam in [0.0, 0.5, 1.0]:
    ie = ImportanceEngine(lambda_uncertainty=lam); re = ResolutionEngine(); mp = VariableResolutionMapper2_5D(ie, re)
    mr = mp.map_scene(base_regs)
    tests.append({"variant": f"lambda:{lam}", "mean_I": round(float(np.mean([s.safe_importance for s in mr.importance])), 3),
                  "cells": len(mr.cells), "mean_res": round(float(np.mean([c.resolution for c in mr.cells])), 3)})
# (c) threshold variants
for tf in [0.65, 0.75, 0.85]:
    ie = ImportanceEngine(); re = ResolutionEngine(t_fine=tf, t_med_fine=0.50, t_med_coarse=0.25); mp = VariableResolutionMapper2_5D(ie, re)
    mr = mp.map_scene(base_regs)
    tests.append({"variant": f"thresh_fine:{tf}", "mean_I": round(float(np.mean([s.safe_importance for s in mr.importance])), 3),
                  "cells": len(mr.cells), "mean_res": round(float(np.mean([c.resolution for c in mr.cells])), 3)})
df_sens = pd.DataFrame(tests)
print("[PROTOTYPE SENSITIVITY EXPERIMENTS — synthetic]")
print(df_sens.to_string(index=False))
plt.figure(figsize=(8, 3.5)); plt.bar(df_sens["variant"], df_sens["cells"], color="#17becf")
plt.xticks(rotation=25, ha="right"); plt.ylabel("# cells"); plt.title("Sensitivity: total cells per variant (synthetic)")
plt.tight_layout(); plt.savefig(VIZ_DIR/"sensitivity_cells.png", dpi=150); plt.show()
print("[CHECK] Sensitivity analysis complete.")


## 18. Save Results


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True); VIZ_DIR.mkdir(parents=True, exist_ok=True)
df_scen.to_csv(RESULTS_DIR/"synthetic_scenarios.csv", index=False)
df_bench.to_csv(RESULTS_DIR/"benchmark_results.csv", index=False)
pd.DataFrame([asdict(s) for s in imp_results]).to_csv(RESULTS_DIR/"importance_results.csv", index=False)
config_out = {"MAX_RANGE_M": MAX_RANGE_M, "RESOLUTION_LEVELS": RESOLUTION_LEVELS,
    "thresholds": {"fine": THRESH_FINE, "med_fine": THRESH_MED_FINE, "med_coarse": THRESH_MED_COARSE},
    "weights": W, "lambda": LAMBDA_UNCERTAINTY, "semantic_importance": SEMANTIC_IMPORTANCE,
    "seed": SEED, "note": "initial prototype parameters; synthetic stage only"}
(RESULTS_DIR/"configuration.json").write_text(json.dumps(config_out, indent=2))
print("Saved: synthetic_scenarios.csv, benchmark_results.csv, importance_results.csv, configuration.json")
print(f"Visualizations in {VIZ_DIR} ({len(list(VIZ_DIR.glob('*.png')))} PNGs)")
print("[CHECK] Results saved.")


## 19. Export Reusable Source Modules


In [ ]:
SRC_DIR = Path("src"); SRC_DIR.mkdir(parents=True, exist_ok=True)
# data_types.py
(SRC_DIR/"data_types.py").write_text(textwrap.dedent("""\
    from dataclasses import dataclass, asdict
    import numpy as np
    @dataclass
    class SyntheticRegion:
        region_id: int
        semantic_class: str
        points: np.ndarray
        distance_m: float
        semantic_importance: float
        terrain_complexity: float
        dynamic_relevance: float
        uncertainty: float
        confidence: float
        def __post_init__(self):
            assert self.points.ndim == 2 and self.points.shape[1] == 4
            assert self.distance_m >= 0
            for n in ['semantic_importance','terrain_complexity','dynamic_relevance','uncertainty','confidence']:
                v = getattr(self, n); assert 0.0 <= v <= 1.0
    @dataclass
    class ImportanceResult:
        region_id: int; distance_score: float; semantic_score: float; terrain_score: float
        dynamic_score: float; uncertainty_score: float; base_importance: float; safe_importance: float
        selected_resolution_m: float; resolution_level: str
    @dataclass
    class MapCell:
        x: float; y: float; elevation: float; elevation_std: float; occupancy: int
        semantic_class: str; confidence: float; importance: float; resolution: float
        region_id: int; point_count: int
    @dataclass
    class MapResult:
        cells: list; importance: list; elapsed_s: float; n_points: int
        def to_dataframe(self):
            import pandas as pd
            return pd.DataFrame([asdict(c) for c in self.cells])
    """))
# importance_engine.py
(SRC_DIR/"importance_engine.py").write_text(textwrap.dedent(f"""\
    from .data_types import ImportanceResult
    WEIGHTS = {W!r}; MAX_RANGE_M = {MAX_RANGE_M}; LAMBDA = {LAMBDA_UNCERTAINTY}
    class ImportanceEngine:
        def __init__(self, weights=WEIGHTS, max_range_m=MAX_RANGE_M, lambda_uncertainty=LAMBDA):
            assert max_range_m > 0; assert all(v >= 0 for v in weights.values())
            assert abs(sum(weights.values()) - 1.0) < 1e-9
            self.w = dict(weights); self.max_range_m = max_range_m; self.lam = lambda_uncertainty
        @staticmethod
        def _clip01(v): return float(min(1.0, max(0.0, v)))
        def distance_score(self, d):
            import numpy as np
            assert np.isfinite(d) and d >= 0
            return self._clip01(1.0 - d / self.max_range_m)
        def score_region(self, r):
            D = self.distance_score(r.distance_m); S = self._clip01(r.semantic_importance)
            T = self._clip01(r.terrain_complexity); M = self._clip01(r.dynamic_relevance); U = self._clip01(r.uncertainty)
            base = self.w['distance']*D + self.w['semantic']*S + self.w['terrain']*T + self.w['dynamic']*M + self.w['uncertainty']*U
            return ImportanceResult(r.region_id, D, S, T, M, U, self._clip01(base), min(1.0, base + self.lam*U), -1.0, '')
    """))
# resolution_engine.py
(SRC_DIR/"resolution_engine.py").write_text(textwrap.dedent(f"""\
    LEVELS = {RESOLUTION_LEVELS!r}
    class ResolutionEngine:
        def __init__(self, t_fine={THRESH_FINE}, t_med_fine={THRESH_MED_FINE}, t_med_coarse={THRESH_MED_COARSE}, levels=LEVELS):
            assert t_fine > t_med_fine > t_med_coarse > 0
            self.t_fine, self.t_med_fine, self.t_med_coarse = t_fine, t_med_fine, t_med_coarse
            self.levels = dict(levels)
        def select(self, importance):
            import numpy as np
            assert np.isfinite(importance) and 0.0 <= importance <= 1.0
            if importance >= self.t_fine: return self.levels['fine'], 'fine (5 cm)'
            if importance >= self.t_med_fine: return self.levels['medium_fine'], 'medium_fine (10 cm)'
            if importance >= self.t_med_coarse: return self.levels['medium_coarse'], 'medium_coarse (20 cm)'
            return self.levels['coarse'], 'coarse (50 cm)'
    """))
# synthetic_scene.py (real implementation: controlled geometric primitives + noise + dropout)
(SRC_DIR/"synthetic_scene.py").write_text(textwrap.dedent("""    import numpy as np
    class SyntheticSceneGenerator:
        # Controlled synthetic LiDAR-like primitives (synthetic input only, NOT real LiDAR).
        def __init__(self, seed=42, dropout=0.02):
            assert 0.0 <= dropout < 1.0
            self.rng = np.random.default_rng(seed); self.dropout = dropout
        def _dropout(self, pts):
            if self.dropout <= 0 or len(pts) == 0: return pts
            return pts[self.rng.random(len(pts)) >= self.dropout]
        def _finish(self, xyz, mu, sigma):
            inten = np.clip(self.rng.normal(mu, sigma, len(xyz)), 0, 1)
            return self._dropout(np.column_stack([xyz, inten]))
        def generate_region_points(self, spec):
            kind = spec['kind']; cx, cy, cz = spec['center']
            n = int(spec.get('n_points', 1000)); noise = float(spec.get('noise', 0.02))
            sx, sy = spec['size'][0], spec['size'][1]
            sz = spec['size'][2] if len(spec['size']) > 2 else 1.0
            R = self.rng
            if kind == 'plane':
                xyz = np.column_stack([R.uniform(cx-sx/2, cx+sx/2, n), R.uniform(cy-sy/2, cy+sy/2, n),
                                       cz + R.normal(0, noise, n)])
                return self._finish(xyz, 0.35, 0.08)
            if kind == 'box':
                top = max(1, n//3); xs, ys, zs = [], [], []
                for _ in range(top):
                    xs.append(R.uniform(cx-sx/2, cx+sx/2)); ys.append(R.uniform(cy-sy/2, cy+sy/2)); zs.append(cz+sz/2)
                for _ in range(n - top):
                    side = R.integers(0, 4)
                    if side == 0: xs.append(cx+sx/2); ys.append(R.uniform(cy-sy/2, cy+sy/2)); zs.append(R.uniform(cz-sz/2, cz+sz/2))
                    elif side == 1: xs.append(cx-sx/2); ys.append(R.uniform(cy-sy/2, cy+sy/2)); zs.append(R.uniform(cz-sz/2, cz+sz/2))
                    elif side == 2: xs.append(R.uniform(cx-sx/2, cx+sx/2)); ys.append(cy+sy/2); zs.append(R.uniform(cz-sz/2, cz+sz/2))
                    else: xs.append(R.uniform(cx-sx/2, cx+sx/2)); ys.append(cy-sy/2); zs.append(R.uniform(cz-sz/2, cz+sz/2))
                xyz = np.column_stack([xs, ys, zs]) + R.normal(0, noise, (n, 3))
                return self._finish(xyz, 0.6, 0.12)
            if kind == 'pedestrian':
                xyz = np.column_stack([cx + R.normal(0, sx*0.35, n), cy + R.normal(0, sy*0.35, n),
                                       R.normal(cz, sz*0.28, n)]) + R.normal(0, noise, (n, 3))
                return self._finish(xyz, 0.55, 0.1)
            if kind == 'wall':
                xyz = np.column_stack([R.uniform(cx-sx/2, cx+sx/2, n),
                                       cy + R.normal(0, max(noise, sy*0.1), n),
                                       R.uniform(cz-sz/2, cz+sz/2, n)]) + R.normal(0, noise, (n, 3))
                return self._finish(xyz, 0.5, 0.1)
            if kind == 'scatter':
                xyz = np.column_stack([R.normal(cx, sx*0.25, n), R.normal(cy, sy*0.25, n),
                                       np.abs(R.normal(cz*0.5, sz*0.3, n))]) + R.normal(0, noise, (n, 3))
                return self._finish(xyz, 0.3, 0.12)
            if kind == 'rough':
                x = R.uniform(cx-sx/2, cx+sx/2, n); y = R.uniform(cy-sy/2, cy+sy/2, n)
                amp = spec.get('rough_amp', 0.35)
                z = cz + amp*np.sin(0.8*x)*np.cos(0.9*y) + R.normal(0, noise + 0.08, n)
                return self._finish(np.column_stack([x, y, z]), 0.32, 0.08)
            raise ValueError(f'unknown primitive kind: {kind}')
        def generate_scene(self, specs):
            return {int(s['region_id']): self.generate_region_points(s) for s in specs}
    """))
# mapper_2_5d.py + evaluation.py (functional minimal cores re-exporting notebook logic)
(SRC_DIR/"mapper_2_5d.py").write_text(textwrap.dedent("""\
    import time, numpy as np
    from .data_types import MapCell, MapResult
    class VariableResolutionMapper2_5D:
        def __init__(self, imp_engine, res_engine):
            self.imp = imp_engine; self.res = res_engine
        def map_region(self, region, importance):
            res_m, _ = self.res.select(importance)
            pts = region.points
            ix = np.floor(pts[:,0]/res_m).astype(np.int64)
            iy = np.floor(pts[:,1]/res_m).astype(np.int64)
            ukeys, inv = np.unique(np.column_stack([ix,iy]), axis=0, return_inverse=True)
            cells = []
            for k,(cx,cy) in enumerate(ukeys):
                z = pts[inv==k,2]
                cells.append(MapCell(float((cx+0.5)*res_m), float((cy+0.5)*res_m), float(z.mean()),
                    float(z.std() if len(z)>1 else 0.0), 1, region.semantic_class, region.confidence,
                    float(importance), float(res_m), region.region_id, int((inv==k).sum())))
            return cells
        def map_scene(self, regs):
            t0=time.perf_counter(); cells=[]; impl=[]
            n=0
            for r in regs:
                s=self.imp.score_region(r); rm,lvl=self.res.select(s.safe_importance)
                s.selected_resolution_m, s.resolution_level = rm, lvl
                cells+=self.map_region(r,s.safe_importance); impl.append(s); n+=len(r.points)
            return MapResult(cells, impl, time.perf_counter()-t0, n)
    """))
(SRC_DIR/"evaluation.py").write_text(textwrap.dedent("""\
    import numpy as np
    DIST_BINS=[(10.0,0.05),(30.0,0.10),(60.0,0.20),(float('inf'),0.50)]
    def distance_resolution(d):
        for t,r in DIST_BINS:
            if d<=t: return r
        return 0.50
    """))
(SRC_DIR/"__init__.py").write_text("")
print(f"Exported {[p.name for p in SRC_DIR.glob('*.py')]}")
# Smoke test: import back
import sys; sys.path.insert(0, ".")
from src.data_types import MapCell as MC2
from src.resolution_engine import ResolutionEngine as RE2
from src.importance_engine import ImportanceEngine as IE2
from src.mapper_2_5d import VariableResolutionMapper2_5D as MP2
from src.synthetic_scene import SyntheticSceneGenerator as SG2
assert RE2().select(0.9)[0] == 0.05 and RE2().select(0.1)[0] == 0.50
sg = SG2(seed=SEED).generate_region_points(DEFAULT_REGION_SPECS[0])
assert sg.ndim == 2 and sg.shape[1] == 4 and len(sg) > 0, "exported generator must return N x 4"
m = MP2(IE2(), RE2()).map_scene(regions)
assert len(m.cells) > 0 and len(set(c.resolution for c in m.cells)) > 1
print(f"[CHECK] Smoke test passed: src/ modules importable, {len(m.cells)} cells, multi-resolution OK.")


## 20. Final Validation


In [ ]:
checks = []
def v(name, cond):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}"); checks.append(bool(cond))
v("Synthetic scene generated", len(clouds) == len(DEFAULT_REGION_SPECS) and all(p.shape[1]==4 for p in clouds.values()))
v("Region features generated", len(region_features) == len(DEFAULT_REGION_SPECS))
v("Importance Engine working", all(0 <= s.safe_importance <= 1 for s in imp_results))
v("Resolution Engine working", all(c.resolution in (0.05,0.10,0.20,0.50) for c in map_result.cells))
v("Adaptive 2.5D Mapper working", len(map_result.cells) > 0 and all(hasattr(c,'elevation') for c in map_result.cells))
v("Different resolutions actually created", len(set(c.resolution for c in map_result.cells)) > 1)
v("Core USP demonstrated", ped_res < road_res)
v("Baselines implemented", len(df_bench) == 3)
v("Synthetic benchmark completed", len(df_perf) == 3)
v("Results saved", (RESULTS_DIR/"configuration.json").is_file() and (RESULTS_DIR/"benchmark_results.csv").is_file())
v("Source modules exported", Path("src/mapper_2_5d.py").is_file() and Path("src/importance_engine.py").is_file())
print(f"\nValidation: {sum(checks)}/{len(checks)} passed")
assert all(checks), "final validation failed"
print("\n====================================================" \
"\nSTAGE 1 SYNTHETIC PROTOTYPE COMPLETE" \
"\n====================================================" \
"\nCore algorithm:\n    VERIFIED" \
"\nImportance Engine:\n    VERIFIED" \
"\nResolution Engine:\n    VERIFIED" \
"\nAdaptive 2.5D Mapper:\n    VERIFIED" \
"\nCore USP demonstration:\n    VERIFIED" \
"\nReal dataset:\n    NOT USED IN THIS STAGE" \
"\nReady for:\n    Small-scale real nuScenes integration" \
"\n====================================================")


## 21. Future Real-LiDAR Integration
**Current stage:** synthetic LiDAR-like input (`SyntheticSceneGenerator` + hand-defined simulated perception).

**Future stage:** `nuScenes LIDAR_TOP` input via a `NuScenesLiDARLoader` that outputs the same `SyntheticRegion`-compatible features (region_id, semantic_class, points Nx4, distance_m, semantic_importance, terrain_complexity, dynamic_relevance, uncertainty, confidence).

Downstream interface stays unchanged:

```text
Input features -> Importance Engine -> Resolution Engine -> Adaptive 2.5D Mapper
```

Only the front-end is replaced:

```text
SyntheticSceneGenerator  -->  NuScenesLiDARLoader
```

with NO rewrite of the core Importance Engine, Resolution Engine, or Mapper. Real-world validation, latency, and accuracy can only be claimed after that stage.
